# Stage 6 & 7: Algorithm 2 — Random Forest Classifier

**Owner:** Member B  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/random_forest_tuned.pkl`  

### Scope
This notebook implements the tree bagging ensemble model family for late delivery risk prediction.
It evaluates single Decision Trees, baseline Random Forest, performs 5-Fold Stratified K-Fold hyperparameter tuning on `X_train`, evaluates performance on `X_val`, and exports the tuned model artifact.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report
)

FINAL_DIR = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}')


X_train shape: (67529, 67), X_val shape: (14470, 67)


In [2]:
# ── Decision Tree Baseline ─────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
dt_probs = dt.predict_proba(X_val)[:, 1]
print(f'Decision Tree Val ROC-AUC : {roc_auc_score(y_val, dt_probs):.4f}')
print(f'Decision Tree Val PR-AUC  : {average_precision_score(y_val, dt_probs):.4f}')


Decision Tree Val ROC-AUC : 0.6459
Decision Tree Val PR-AUC  : 0.0855


In [4]:
# ── 5-Fold Stratified RandomizedSearchCV ──────────────────────────────────
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=10, scoring='average_precision', cv=cv, random_state=42, n_jobs=-1
)
rand.fit(X_train, y_train)

print('Best Hyperparameters:', rand.best_params_)
print(f'Best Stratified CV PR-AUC: {rand.best_score_:.4f}')

best_rf = rand.best_estimator_
joblib.dump(best_rf, MODELS_DIR / 'random_forest_tuned.pkl')
print('Saved model artifact to models/random_forest_tuned.pkl')


KeyboardInterrupt: 